# PyTorch Experience · 02
## `model(x)` · `model.forward(x)` · `model.eval()` · Hooks

**At a glance:** `model(x)` performs the forward computation with PyTorch's module-call handling. `model.forward(x)` calls the method directly. `model.eval()` **changes the model's mode but does not run a forward pass**.

**Official docs:** [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) · [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [register_forward_hook()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)

[← Pytorch Experience](https://lawson-dong.vercel.app/coding/pytorch-experience)

### 1 · Three calls, three roles

![Diagram comparing model(x) and model.forward(x)](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-model-call-vs-forward.svg)

| Call | What it does | Runs a forward pass? |
|---|---|---|
| `model(x)` | Calls `forward()` through `nn.Module` (including hooks) | **Yes** |
| `model.forward(x)` | Calls `forward()` directly, skipping hooks on this module | **Yes** |
| `model.eval()` | Switches to evaluation mode | **No** |

**Rule:** Define `forward()`; use `model(x)` to calculate; use `model.eval()` to choose evaluation behavior.

### 2 · Two calls, same calculation

This simple model doubles a tensor. Run the cell—no GPU or downloads needed.

In [ ]:
import torch
from torch import nn

class Double(nn.Module):
    def forward(self, x):
        return x * 2

model = Double()
x = torch.tensor([1., 2.])

print('model(x):        ', model(x))
print('model.forward(x):', model.forward(x))

### 3 · Add a forward hook

A **forward hook** is a function PyTorch calls **after** a module's `forward()` has run. It receives `(module, inputs, output)`.

You can use it to **observe/save an output without editing `forward()`**. A hook *can* also replace an output if it explicitly returns a replacement, but the hook below only prints.

In [ ]:
def show_hook(module, inputs, output):
    print('HOOK ran! Output:', output)

handle = model.register_forward_hook(show_hook)

print('1) model(x)')
model(x)            # Hook runs

print('2) model.forward(x)')
model.forward(x)    # Hook on model does NOT run

### 4 · Remove the hook

Hooks stay registered until you remove them. Calling `handle.remove()` cleans up.

In [ ]:
handle.remove()
print('After removing the hook:')
model(x)  # No hook message

### 5 · `model.eval()` changes the mode

`eval()` changes the behavior of layers such as **Dropout** and **BatchNorm**. It does **not** run the model, change the learned weights, or disable gradients.

To make the difference impossible to miss, this example uses `Dropout(p=1.0)` (drops **all** values *in training mode only*).

In [ ]:
model = nn.Sequential(nn.Dropout(p=1.0))  # extreme p just for a clear example
x = torch.ones(4)

model.train()
print('Training:', model(x))  # tensor([0., 0., 0., 0.])

model.eval()
print('Eval:    ', model(x))  # tensor([1., 1., 1., 1.])
print('model.training:', model.training)  # False

### 6 · Evaluation mode vs inference mode

```python
model.eval()                      # changes Dropout / BatchNorm behavior
with torch.inference_mode():     # no autograd bookkeeping
    output = model(x)            # actually computes the output
```

**They are not interchangeable.** `model.eval()` does not turn off gradient tracking; `torch.inference_mode()` does not automatically switch a model to evaluation mode.

**Official docs:** [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [Module.train()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) · [torch.inference_mode()](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)

### Remember

- **`model(x)`** → performs a forward pass and handles the module's registered hooks.
- **`model.forward(x)`** → performs a direct forward call; skips hooks registered on *this* module (submodules called inside it may still run their own hooks).
- **`model.eval()`** → switches mode; **does not** compute an output.
- **Forward hook** → runs after a module's forward pass, for observing outputs (or optionally replacing an output).
- **`torch.inference_mode()`** → controls autograd bookkeeping, not training/evaluation mode.

**References:** [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) · [register_forward_hook()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook) · [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [torch.inference_mode()](https://docs.pytorch.org/docs/stable/generated/torch.autograd.grad_mode.inference_mode.html)